# 04 · Feature engineering

Can new columns built from the existing ones improve the models?

Each candidate feature is measured with the same 5-fold CV as notebook 03 (same folds, ROC-AUC) and compared with the baseline **fold by fold**. The rule for keeping a feature is written down before the experiment runs.

In [1]:
import numpy as np 
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_validate

from credit_risk_ai.data.load import load_train
from credit_risk_ai.data.clean import clean, drop_duplicate_rows
from credit_risk_ai.data.features import add_features
from credit_risk_ai.models.pipeline import build_logistic_regression, build_hist_gradient_boosting

N_SPLITS = 5
RANDOM_STATE = 42
TARGET = "SeriousDlqin2yrs"


In [2]:
data = clean(drop_duplicate_rows(load_train()))

X_base = data.drop(columns=[TARGET])
y = data[TARGET]

X_base.shape 

(149391, 14)

In [3]:
cv = StratifiedKFold(n_splits = N_SPLITS, shuffle = True, random_state=RANDOM_STATE)


In [4]:
builders = {
    "logistic_regression": build_logistic_regression,
    "hist_gradient_boosting": build_hist_gradient_boosting,
}

baseline = {}
for name, build in builders.items():
    baseline[name] = cross_validate(build(), X_base, y, cv=cv, scoring="roc_auc", n_jobs=-1)["test_score"]

for name, scores in baseline.items():
    print(f"{name}: {scores.mean():.4f}")

logistic_regression: 0.8551
hist_gradient_boosting: 0.8649


## Step 3 · Hypotheses and decision rule

| Feature | Formula | Why it might help |
|---|---|---|
| `total_past_due` | 30-59 + 60-89 + 90 days late | Total number of late payments, whatever the length |
| `weighted_past_due` | 1×(30-59) + 2×(60-89) + 3×(90) | Longer delays are more serious |
| `monthly_debt` | DebtRatio × MonthlyIncome | Debt payments in dollars, not as a ratio |
| `income_per_person` | MonthlyIncome / (NumberOfDependents + 1) | Income shared by the whole household |
| `non_real_estate_lines` | OpenCreditLines − RealEstateLoans | Credit cards and personal loans only |

All five are computed from a single row, like `clean()`, so they learn nothing from other rows and cannot leak information between folds.

**Decision rule (fixed before running the experiment):** keep a feature for a model if it improves that model's ROC-AUC on **at least 4 of 5 folds** AND the **mean gain is at least +0.001**. Smaller gains are too small to matter and not worth the extra complexity.

In [5]:
from credit_risk_ai.data.clean import PAST_DUE_COLS
def add_candidate_features(df):
    out = df.copy()
    late_30, late_60, late_90 = (out[col] for col in PAST_DUE_COLS)

    out["total_past_due"] = late_30 + late_60 + late_90
    out["weighted_past_due"] = 1 * late_30 + 2*late_60 + 3*late_90
    out["monthly_debt"] = out["DebtRatio"] * out["MonthlyIncome"]
    out["income_per_person"] = out["MonthlyIncome"] / (out["NumberOfDependents"] + 1)
    out["non_real_estate_lines"] = out["NumberOfOpenCreditLinesAndLoans"] - out["NumberRealEstateLoansOrLines"]

    return out

CANDIDATES = [
    "total_past_due",
    "weighted_past_due",
    "monthly_debt",
    "income_per_person",
    "non_real_estate_lines",
]

X_all = add_candidate_features(X_base)

In [6]:
assert set(X_all.columns) - set(X_base.columns) == set(CANDIDATES)

In [7]:
print("X_all", X_all.shape, "| X_base:", X_base.shape)

pd.DataFrame({
    "missing": X_all[CANDIDATES].isna().sum(),
    "min": X_all[CANDIDATES].min(),
    "median": X_all[CANDIDATES].median(),
    "max": X_all[CANDIDATES].max(),
}).round(2)

X_all (149391, 19) | X_base: (149391, 14)


,missing,min,median,max
total_past_due,225,0.00,0.00,19.00
weighted_past_due,225,0.00,0.00,51.00
monthly_debt,31433,0.00,1658.56,478450.56
income_per_person,31433,0.67,3400.00,1794060.00
non_real_estate_lines,0,0.00,7.00,57.00


In [8]:
experiments = {feature: [feature] for feature in CANDIDATES}
experiments["all"] = CANDIDATES

rows = []
for exp_name, cols in experiments.items():
    X = X_all[list(X_base.columns) + cols]
    for model_name, build in builders.items():
        scores = cross_validate(build(), X, y, cv=cv, scoring="roc_auc", n_jobs=-1)["test_score"]
        diff = scores - baseline[model_name]
        rows.append({
            "feature": exp_name,
            "model": model_name,
            "roc_auc": scores.mean(),
            "gain": diff.mean(),          # average change vs baseline
            "wins": (diff > 0).sum(),     # folds where the feature helped
            "worst": diff.min(),          # worst fold: negative = it hurt on that fold
        })

results = pd.DataFrame(rows).round(4)
results

,feature,model,roc_auc,gain,wins,worst
0,total_past_due,logistic_regression,0.8577,0.0026,5,0.0019
1,total_past_due,hist_gradient_boosting,0.8649,0.0001,3,-0.0008
2,weighted_past_due,logistic_regression,0.8582,0.0031,5,0.0020
3,weighted_past_due,hist_gradient_boosting,0.8651,0.0002,3,-0.0009
4,monthly_debt,logistic_regression,0.8554,0.0003,5,0.0002
5,monthly_debt,hist_gradient_boosting,0.8652,0.0003,4,-0.0002
6,income_per_person,logistic_regression,0.8551,-0.0000,1,-0.0001
7,income_per_person,hist_gradient_boosting,0.8648,-0.0001,2,-0.0005
8,non_real_estate_lines,logistic_regression,0.8551,-0.0000,3,-0.0000
9,non_real_estate_lines,hist_gradient_boosting,0.8649,0.0000,2,-0.0007


In [9]:
results["keep"] = (results["wins"] >= 4 ) & (results["gain"] >= 0.001)
results.pivot(index="feature", columns="model", values="keep")

model,hist_gradient_boosting,logistic_regression
feature,,
all,False,True
income_per_person,False,False
monthly_debt,False,False
non_real_estate_lines,False,False
total_past_due,False,True
weighted_past_due,False,True


## Step 7 · Final feature set from `src`

Decision from step 6: keep only **`weighted_past_due`**.

- It passes the rule for logistic regression (+0.0031, 5/5 folds), almost as much as all five candidates together (+0.0033).
- `total_past_due` also passes, but it carries nearly the same information, so only one of the two is kept.
- No feature passes for HistGradientBoosting; `weighted_past_due` does not hurt it, so both models share one feature set.

The feature now lives in `src/credit_risk_ai/data/features.py` (`add_features`, tested in `tests/test_features.py`).
This step checks that the `src` version gives exactly the same column and the same CV scores as the experiment above.

In [10]:
X_final = add_features(X_base)

# The src function must give exactly the column tested in the experiment (step 4)
pd.testing.assert_series_equal(X_final["weighted_past_due"], X_all["weighted_past_due"])
print("X_final:", X_final.shape)

final_rows = []
for model_name, build in builders.items():
    scores = cross_validate(build(), X_final, y, cv=cv, scoring="roc_auc", n_jobs=-1)["test_score"]
    diff = scores - baseline[model_name]          # paired: same folds as the baseline
    final_rows.append({
        "model": model_name,
        "baseline": baseline[model_name].mean(),
        "with_feature": scores.mean(),
        "gain": diff.mean(),
        "wins": (diff > 0).sum(),
        "worst": diff.min(),
    })

final = pd.DataFrame(final_rows).set_index("model").round(4)
final

X_final: (149391, 15)


,baseline,with_feature,gain,wins,worst
model,,,,,
logistic_regression,0.8551,0.8582,0.0031,5,0.0020
hist_gradient_boosting,0.8649,0.8651,0.0002,3,-0.0009


## Step 8 · Conclusions

**Setup:** 5 candidate features, each added on its own (and all together), scored with the same 5-fold CV as notebook 03 and compared with the baseline fold by fold.

| Model | Baseline | With `weighted_past_due` | Gain | Folds improved |
|---|---|---|---|---|
| Logistic regression | 0.8551 | **0.8582** | +0.0031 | 5 / 5 |
| HistGradientBoosting | 0.8649 | 0.8651 | +0.0002 | 3 / 5 |

**Findings**

1. **Most ideas did not help.** `income_per_person` and `non_real_estate_lines` changed nothing. `monthly_debt` helped consistently (4–5 of 5 folds) but only by +0.0003, which is below the +0.001 bar: consistent is not the same as useful.
2. **Past-due features help logistic regression, not HistGradientBoosting.** A linear model cannot add the three past-due counts together on its own once each one is log-transformed, so giving it the combined count helps (+0.003 on every fold). Trees can already combine these columns through successive splits, so HistGradientBoosting gains nothing (changes of ±0.0002 are noise).
3. **One feature is enough.** `total_past_due` and `weighted_past_due` carry almost the same information; all five together (+0.0033) are barely better than `weighted_past_due` alone (+0.0031).

**Decision:** keep only `weighted_past_due`, moved to `src/credit_risk_ai/data/features.py` (`add_features`, applied after `clean()`) and covered by `tests/test_features.py`. The `src` version reproduces the experiment exactly (same column, same CV scores).

**What this means:** the main model (HistGradientBoosting) stays at about **0.865**; feature engineering is not where its next gain will come from.

**Next step (notebook 05):** tune HistGradientBoosting on `add_features(...)` data with `RandomizedSearchCV` and the same `cv`, aiming to reduce its train/validation gap (0.021 in notebook 03).